In [4]:
import pandas as pd
import requests
import json

from google_play_scraper import Sort, reviews

In [5]:
result, continuation_token = reviews(
    'id.co.bri.brimo', # application unique tag in google playstore
    lang='id', # defaults to 'en'
    country='id', # defaults to 'us'
    sort=Sort.NEWEST, # defaults to Sort.NEWEST
    count=20, # defaults to 100
    filter_score_with=None # defaults to None(means all score)
)

df = pd.DataFrame(result)
df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,8d4330da-1eff-4fc6-b986-d15a59ce159e,Iyus Tini,https://play-lh.googleusercontent.com/a/ACg8oc...,Nyaman dan aman,5,0,2.63.0,2024-07-15 16:45:52,Terima kasih atas ulasannya. Semoga aplikasi B...,2024-07-15 17:10:28,2.63.0
1,d9bfc16b-2397-4259-bec2-7ddb71da0a87,Wahyudi Traknus,https://play-lh.googleusercontent.com/a-/ALV-U...,"Hore, akhirnya saya punya rekening & akun Bank...",5,0,2.63.0,2024-07-15 16:40:49,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 18:52:14,2.63.0
2,79f276a5-14bd-4af0-9c74-73562a5e7416,Ana Nia,https://play-lh.googleusercontent.com/a/ACg8oc...,Akhir2 ini kok susah yaa brimo nya kalo mau tf...,4,0,2.63.0,2024-07-15 16:32:26,None,NaT,2.63.0
3,1ac9f80f-9ad9-48bd-a730-61f62a4769b7,Akmal Akmal,https://play-lh.googleusercontent.com/a/ACg8oc...,lebih simpel pakenya,5,0,2.61.0,2024-07-15 16:10:19,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:18,2.61.0
4,61640f8b-9c16-4af2-bf58-0629980dd70d,Aulana Pramada,https://play-lh.googleusercontent.com/a/ACg8oc...,Mantap,5,0,2.63.0,2024-07-15 16:10:16,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:16,2.63.0
5,6392a95e-5c9a-45b0-ab2c-47b008ec121b,Ariff Setiawann,https://play-lh.googleusercontent.com/a/ACg8oc...,Jadi lebih gapang,5,0,2.52.0,2024-07-15 16:09:53,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:14,2.52.0
6,ac5a69fc-9bf0-4ea1-9d61-20d897058f63,Merry Maniku,https://play-lh.googleusercontent.com/a/ACg8oc...,Cepat and mudah,5,0,2.63.0,2024-07-15 16:08:54,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:13,2.63.0
7,e9dff493-5f30-42f3-af86-0b6c66af3d9d,Khoiron M,https://play-lh.googleusercontent.com/a-/ALV-U...,Kenpa punya saya gak bisa masuk ya skrng pasti...,1,0,2.63.0,2024-07-15 16:05:27,"Sobat BRI, mohon maaf atas ketidaknyamanan yan...",2024-07-15 19:51:21,2.63.0
8,11a7c228-1a77-4907-a66a-4a84a35a3837,Rama prayoga,https://play-lh.googleusercontent.com/a-/ALV-U...,Okeee,5,0,None,2024-07-15 16:04:15,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:10,None
9,582a89e7-6ff0-48e1-802d-b1204a11db65,Nurazzahra Widhi,https://play-lh.googleusercontent.com/a-/ALV-U...,Bs dg cpt bertansaksi,5,0,2.63.0,2024-07-15 16:03:48,"Hai Sobat BRI, terima kasih atas ulasannya. Se...",2024-07-15 16:17:09,2.63.0


In [6]:
def review_analyze(text):
    payload = {
        "model" : "llama3",
        "messages" : [
            {"role" : "system",
             "content":f"""
                You are an expert in analyzing user reviews. Identify the key topics and sentiments of the following Playstore review for the BRImo mobile banking app by Bank Rakyat Indonesia, one of Indonesia's most reputable banks. Summarize the review and provide its sentiment.
                 """},
            {"role" : "user",
             "content": f"""
                Question: Identify the key topics of this playstore review for a mobile banking app of BRI (Bank Rakyat Indonesia) 
                          called BRImo which is one of the most reputable bank in Indonesia and what is the sentiments of these review?

                Review: "{text}"   

                Thought 1: Understand the content of the review and identify its key topics.
                Action 1: Analyze the review text to extract key topics.
                Observation 1: The key topics are...
                
                Thought 2: Analyze the tone and language of the review.
                Action 2: Determine the sentiment of the review text.
                Observation 2: The review text indicates positive/negative/neutral sentiment.

                Conclusion: Summarize the key topics and sentiment of the review using this format AND NOTHING ELSE BESIDES THIS: 
                - Topic (1 or 2 words that describe the topic of the review, Fill with 'Overall' if the review doesn't mention anything specifically)
                - Sentiment (Negative, Neutral, or Positive)
                - Explanation (1 to 2 sentence, explain briefly the analysis) 

                Return your answer in JSON format.
                """}
        ] 
    }

    analysis_result = ""
    response = requests.post("http://localhost:11434/api/chat", json=payload)
    for message in response.iter_lines():
        jsonstr = json.loads(message)
        llm_answer = jsonstr["message"]["content"]
        analysis_result += llm_answer
    
    return analysis_result

    
results = []
for i, row in df.iterrows():
    review_id = row['reviewId']
    username = row['userName']
    datetime = row['at']
    rating = row['score']
    review_content = row['content']
    analysis = review_analyze(review_content)
    results.append({"review_id": review_id, "username": username, "datetime":datetime, "rating":rating, 
                    "review": review_content, "analysis": analysis})

results_df = pd.DataFrame(results)

In [ ]:
results_df.drop('analysis')

,username,datetime,rating,review,analysis
0,Akmal Akmal,2024-07-15 16:10:19,5,lebih simpel pakenya,"{\n""Topic"": ""Ease"",\n""Sentiment"": ""Positive"",\..."
1,Aulana Pramada,2024-07-15 16:10:16,5,Mantap,"{\n""Topic"": ""Appreciation"",\n""Sentiment"": ""Pos..."
2,Ariff Setiawann,2024-07-15 16:09:53,5,Jadi lebih gapang,"{\n""Topic"": ""Satisfaction"",\n""Sentiment"": ""Pos..."
3,Merry Maniku,2024-07-15 16:08:54,5,Cepat and mudah,"{\n""Topic"": ""User Experience"",\n""Sentiment"": ""..."
4,Khoiron M,2024-07-15 16:05:27,1,Kenpa punya saya gak bisa masuk ya skrng pasti...,"{\n""Topic"": ""Login Issues"",\n""Sentiment"": ""Neg..."
5,Rama prayoga,2024-07-15 16:04:15,5,Okeee,"{\n""Topic"": ""Simple"",\n""Sentiment"": ""Positive""..."
6,Nurazzahra Widhi,2024-07-15 16:03:48,5,Bs dg cpt bertansaksi,"{\n""Topic"": ""Transaction"",\n""Sentiment"": ""Posi..."
7,Ikbal Maulana,2024-07-15 16:01:29,5,Mudah belanja tranfer,"{\n""Topic"": ""Easy Transfer"",\n""Sentiment"": ""Po..."
8,Rudy Karyono,2024-07-15 15:58:43,5,Mantap,"{\n""Topic"": ""Appraisal"",\n""Sentiment"": ""Positi..."
9,AS EF,2024-07-15 15:58:12,4,Alhamdulillah selama penggunaan nya belum ada ...,"{\n""Topic"": ""App Performance"",\n""Sentiment"": ""..."


In [ ]:
print(results_df.iloc[0,:]['analysis'])

{
"Topic": "Ease",
"Sentiment": "Positive",
"Explanation": "The review mentions 'lebih simpel pakenya' which translates to 'it's easier to use', indicating that the reviewer finds the app easy to navigate and use."
}


In [ ]:
# Function to parse the JSON-like string
def parse_json_column(json_str):
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        return {"Topics": None, "Sentiment": None, "Explanation": None}

# Apply the function to parse the column
parsed_column = results_df["analysis"].apply(parse_json_column)

# Convert the parsed column to a DataFrame
parsed_df = pd.json_normalize(parsed_column)

# Concatenate the new DataFrame with the original DataFrame
results_df = pd.concat([results_df, parsed_df], axis=1)
results_df

,username,datetime,rating,review,analysis,Topic,Sentiment,Explanation
0,Akmal Akmal,2024-07-15 16:10:19,5,lebih simpel pakenya,"{\n""Topic"": ""Ease"",\n""Sentiment"": ""Positive"",\...",Ease,Positive,The review mentions 'lebih simpel pakenya' whi...
1,Aulana Pramada,2024-07-15 16:10:16,5,Mantap,"{\n""Topic"": ""Appreciation"",\n""Sentiment"": ""Pos...",Appreciation,Positive,The review only contains a single word 'Mantap...
2,Ariff Setiawann,2024-07-15 16:09:53,5,Jadi lebih gapang,"{\n""Topic"": ""Satisfaction"",\n""Sentiment"": ""Pos...",Satisfaction,Positive,The review contains the phrase 'Jadi lebih gap...
3,Merry Maniku,2024-07-15 16:08:54,5,Cepat and mudah,"{\n""Topic"": ""User Experience"",\n""Sentiment"": ""...",User Experience,Positive,"The review mentions 'Cepat' and 'mudah', which..."
4,Khoiron M,2024-07-15 16:05:27,1,Kenpa punya saya gak bisa masuk ya skrng pasti...,"{\n""Topic"": ""Login Issues"",\n""Sentiment"": ""Neg...",Login Issues,Negative,The reviewer mentions they can't log in and bl...
5,Rama prayoga,2024-07-15 16:04:15,5,Okeee,"{\n""Topic"": ""Simple"",\n""Sentiment"": ""Positive""...",Simple,Positive,The reviewer's brief and positive response sug...
6,Nurazzahra Widhi,2024-07-15 16:03:48,5,Bs dg cpt bertansaksi,"{\n""Topic"": ""Transaction"",\n""Sentiment"": ""Posi...",Transaction,Positive,The review is short and does not provide speci...
7,Ikbal Maulana,2024-07-15 16:01:29,5,Mudah belanja tranfer,"{\n""Topic"": ""Easy Transfer"",\n""Sentiment"": ""Po...",Easy Transfer,Positive,The review mentions that transferring money is...
8,Rudy Karyono,2024-07-15 15:58:43,5,Mantap,"{\n""Topic"": ""Appraisal"",\n""Sentiment"": ""Positi...",Appraisal,Positive,The single-word review 'Mantap' is an Indonesi...
9,AS EF,2024-07-15 15:58:12,4,Alhamdulillah selama penggunaan nya belum ada ...,"{\n""Topic"": ""App Performance"",\n""Sentiment"": ""...",App Performance,Positive,The reviewer mentions that they have not exper...


In [ ]:
results_df.to_csv('results.csv', index=False)